<a href="https://colab.research.google.com/github/guilhermemoraes-lasalle/ColecoesEassociacoes/blob/main/Atividade_Visualizacao_Estatistica_Seaborn_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade Prática — Visualização Estatística de Dados com Seaborn

## Análise de Engajamento em uma Empresa SaaS

Neste notebook será analisado o comportamento de usuários de uma empresa de software por assinatura.

### Objetivos
- analisar a quantidade de usuários por plano;
- visualizar a distribuição de idade;
- comparar tempo de uso entre planos;
- analisar a satisfação dos clientes;
- investigar relações entre uso, satisfação e bugs;
- calcular e visualizar correlações;
- realizar análise multivariada com `pairplot`.


## 1. Importação das Bibliotecas e Configuração


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração de estilo global do Seaborn
sns.set_theme(
    style="whitegrid",
    palette="muted"
)

print("Bibliotecas carregadas com sucesso!")


## 2. Criação da Base de Dados Sintética


In [ ]:
# Geração de dados sintéticos
np.random.seed(101)
n = 300

dados = {
    'idade': np.random.normal(
        35,
        10,
        n
    ).astype(int),

    'plano': np.random.choice(
        ['Gratuito', 'Básico', 'Pro'],
        n,
        p=[0.5, 0.3, 0.2]
    ),

    'tempo_uso_horas': np.random.uniform(
        1,
        50,
        n
    ),

    'bugs_reportados': np.random.poisson(
        2,
        n
    ),

    'satisfacao': np.random.randint(
        1,
        11,
        n
    )
}

df_sistema = pd.DataFrame(dados)

# Regras de negócio sintéticas
df_sistema.loc[
    df_sistema['plano'] == 'Pro',
    'tempo_uso_horas'
] += 15

df_sistema.loc[
    df_sistema['plano'] == 'Pro',
    'satisfacao'
] += 2

df_sistema['satisfacao'] = (
    df_sistema['satisfacao']
    - (df_sistema['bugs_reportados'] * 0.5)
)

df_sistema['satisfacao'] = (
    df_sistema['satisfacao']
    .clip(1, 10)
    .astype(int)
)

df_sistema['idade'] = (
    df_sistema['idade']
    .clip(18, 70)
)

print("Dataset criado com sucesso!")
display(df_sistema.head(10))

print("\nDimensões:", df_sistema.shape)


## Visão Geral dos Dados


In [ ]:
print("Informações do DataFrame:")
df_sistema.info()

print("\nResumo estatístico:")
display(df_sistema.describe())

print("\nQuantidade de usuários por plano:")
display(df_sistema['plano'].value_counts())


# Parte 1 — Distribuições e Contagens


## 3. Countplot — Quantidade de Usuários por Plano


In [ ]:
# Ordem das barras pela frequência
ordem_planos = (
    df_sistema['plano']
    .value_counts()
    .index
)

plt.figure(figsize=(8, 5))

sns.countplot(
    data=df_sistema,
    x='plano',
    order=ordem_planos
)

plt.title('Quantidade de Usuários por Plano')
plt.xlabel('Plano de Assinatura')
plt.ylabel('Quantidade de Usuários')

plt.tight_layout()
plt.show()


### Interpretação

O `countplot` permite comparar diretamente a frequência de usuários em cada plano. As barras foram ordenadas da categoria mais frequente para a menos frequente.


## 4. Histplot — Distribuição da Idade


In [ ]:
plt.figure(figsize=(9, 6))

sns.histplot(
    data=df_sistema,
    x='idade',
    bins=20,
    kde=True
)

plt.title('Distribuição da Idade dos Usuários')
plt.xlabel('Idade')
plt.ylabel('Frequência')

plt.tight_layout()
plt.show()


### Interpretação

O histograma apresenta a frequência das idades, enquanto a curva KDE fornece uma representação suavizada da distribuição.


# Parte 2 — Relações Categóricas e Numéricas


## 5. Boxplot — Tempo de Uso por Plano


In [ ]:
plt.figure(figsize=(9, 6))

sns.boxplot(
    data=df_sistema,
    x='plano',
    y='tempo_uso_horas'
)

plt.title('Distribuição do Tempo de Uso por Plano')
plt.xlabel('Plano de Assinatura')
plt.ylabel('Tempo de Uso (horas)')

plt.tight_layout()
plt.show()


### Interpretação

O boxplot facilita a comparação da mediana, dispersão e possíveis valores extremos do tempo de uso entre os diferentes planos.


## 6. Violinplot — Satisfação por Plano


In [ ]:
plt.figure(figsize=(9, 6))

sns.violinplot(
    data=df_sistema,
    x='plano',
    y='satisfacao',
    inner='box'
)

plt.title('Distribuição da Satisfação por Plano')
plt.xlabel('Plano de Assinatura')
plt.ylabel('Satisfação')

plt.tight_layout()
plt.show()


### Interpretação

O violinplot combina informações de distribuição e densidade, permitindo observar onde as notas de satisfação estão mais concentradas em cada plano.


# Parte 3 — Correlações e Análise Multivariada


## 7. Scatterplot — Tempo de Uso × Satisfação


In [ ]:
plt.figure(figsize=(11, 7))

sns.scatterplot(
    data=df_sistema,
    x='tempo_uso_horas',
    y='satisfacao',
    hue='plano',
    size='bugs_reportados',
    sizes=(30, 250),
    alpha=0.7
)

plt.title(
    'Relação entre Tempo de Uso e Satisfação'
)

plt.xlabel('Tempo de Uso (horas)')
plt.ylabel('Satisfação')

plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)

plt.tight_layout()
plt.show()


### Interpretação

Cada ponto representa um usuário. A cor identifica o plano de assinatura e o tamanho representa a quantidade de bugs reportados. Isso permite analisar várias dimensões simultaneamente.


## 8. Matriz de Correlação de Pearson


In [ ]:
# Selecionando somente variáveis numéricas
variaveis_numericas = df_sistema.select_dtypes(
    include=np.number
)

# Matriz de correlação de Pearson
matriz_correlacao = (
    variaveis_numericas.corr(
        method='pearson'
    )
)

print("Matriz de correlação:")
display(matriz_correlacao.round(2))


## 9. Heatmap da Matriz de Correlação


In [ ]:
plt.figure(figsize=(9, 7))

sns.heatmap(
    matriz_correlacao,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5
)

plt.title(
    'Matriz de Correlação das Variáveis Numéricas'
)

plt.tight_layout()
plt.show()


### Interpretação

Valores próximos de **1** indicam correlação linear positiva, valores próximos de **-1** indicam correlação negativa e valores próximos de **0** indicam pouca associação linear.

Como os dados foram gerados com regras sintéticas, algumas relações foram introduzidas propositalmente para facilitar a análise visual.


# Bônus — Análise Multivariada


## 10. Pairplot por Plano


In [ ]:
sns.pairplot(
    data=df_sistema,
    vars=[
        'tempo_uso_horas',
        'bugs_reportados',
        'satisfacao'
    ],
    hue='plano',
    diag_kind='hist'
)

plt.show()


## 11. Médias por Plano — Análise Complementar


In [ ]:
resumo_planos = (
    df_sistema
    .groupby('plano')[
        [
            'tempo_uso_horas',
            'bugs_reportados',
            'satisfacao'
        ]
    ]
    .mean()
    .round(2)
)

display(resumo_planos)


# Conclusão

Nesta atividade foram utilizados diversos recursos do **Seaborn** para análise estatística visual:

- `countplot` para contagem de categorias;
- `histplot` com KDE para distribuição;
- `boxplot` para comparação de distribuições;
- `violinplot` para análise de densidade;
- `scatterplot` com `hue` e `size`;
- correlação de Pearson;
- `heatmap` com anotações;
- `pairplot` para análise multivariada.

Essas visualizações permitem compreender melhor o perfil de engajamento dos usuários, as diferenças entre os planos e as relações entre tempo de uso, bugs reportados e satisfação.
